# AI-Based Digital Learning Assistant: Baseline Model & Data Exploration

**Course:** MLOps Project  
**Dataset:** ASSISTments Skill Builder  
**Task:** Student Correctness Prediction (Binary Classification)  

---
## 1. Overview & Objectives
This notebook provides the exploratory data inspection, feature preparation, baseline model development, and MLflow experiment tracking for the student correctness prediction module.

The goal is to predict whether a student solves an assigned skill problem correctly on the first attempt (`correct = 1` vs `0`).

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix
)
import mlflow

print('All libraries imported successfully!')

## 2. Load and Inspect Raw Dataset
We load the ASSISTments dataset tracked under `data/raw/` via DVC.

In [ ]:
data_path = '../data/raw/skill_builder_data.csv' if os.path.exists('../data/raw/skill_builder_data.csv') else 'data/raw/skill_builder_data.csv'
df = pd.read_csv(data_path, encoding='latin1', low_memory=False)
print(f'Dataset Shape: {df.shape}')
df.head(5)

In [ ]:
# Inspect target distribution
target_counts = df['correct'].value_counts(dropna=False)
print('Target Distribution [correct]:')
print(target_counts)

# Check missing values for key features
key_features = ['attempt_count', 'ms_first_response', 'hint_count', 'hint_total', 'overlap_time', 'opportunity']
print('\nMissing values in key features:')
print(df[key_features].isnull().sum())

## 3. Data Preprocessing & Train-Test Split
We clean the dataset by:
1. Filtering valid binary targets (`correct` in {0, 1}).
2. Imputing missing values using the column median.
3. Stratifying an 80/20 train/test split.

In [ ]:
# Filter valid target rows
clean_df = df[df['correct'].isin([0, 1])].copy()
clean_df['correct'] = clean_df['correct'].astype(int)

# Impute numerical features with median
for col in key_features:
    clean_df[col] = pd.to_numeric(clean_df[col], errors='coerce')
    clean_df[col] = clean_df[col].fillna(clean_df[col].median())

# Representative sample for fast interactive prototyping (100k rows)
sample_df = clean_df.sample(n=min(100000, len(clean_df)), random_state=42)

X = sample_df[key_features]
y = sample_df['correct']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set shape: {X_train.shape}')
print(f'Testing set shape:  {X_test.shape}')

## 4. Train Baseline Model (Random Forest Classifier)
We train a baseline Random Forest classifier to establish a benchmark for student performance prediction.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Generate predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print('=== Baseline Model Evaluation ===')
print(f'Accuracy:  {acc:.4f}')
print(f'Precision: {prec:.4f}')
print(f'Recall:    {rec:.4f}')
print(f'F1-Score:  {f1:.4f}')
print(f'ROC-AUC:   {roc_auc:.4f}')

## 5. Detailed Classification Report & Feature Importances

In [ ]:
print(classification_report(y_test, y_pred))

# Feature Importance
feat_importances = pd.Series(model.feature_importances_, index=key_features).sort_values(ascending=False)
print('Feature Importances:')
print(feat_importances)

## 6. MLOps Integration (DVC & DAGsHub MLflow)
- **DVC:** The raw data, preprocessed partitions (`data/processed/`), and model artifact (`models/baseline_model.pkl`) are versioned through `dvc.yaml`.
- **MLflow:** Training runs, hyperparameters, and evaluation metrics are logged to DAGsHub at `https://dagshub.com/suhaybfr/ML-ops.mlflow`.